# ONS Data Joins

This notebook details join steps with ONS (Office for National Statistics) data.

By joining the FHRS data to this location-focused ONS data, area factors can be used to help predict the relative risk of a given establisment.  

Two open data files have been downloaded and saved to `../data/raw/`:

__[Lower Layer Super Output Areas](https://open-geography-portalx-ons.hub.arcgis.com/datasets/postcode-to-oa-2021-to-lsoa-to-msoa-to-lad-november-2025-best-fit-lookup-in-the-uk)__ - "A best-fit lookup between postcodes, 2021 Census Output Areas (OA), Lower Layer Super Output Areas (LSOA), Middle Layer Super Output Areas (MSOA) and local authority districts (LAD). Postcodes are as at November 2025 in the UK and are best-fitted by plotting the location of the postcode's mean address into the areas of the output geographies." **File size is approx. 416MB**

__[English indices of deprivation 2025](https://www.gov.uk/government/statistics/english-indices-of-deprivation-2025)__ - "Statistics on relative deprivation in small areas in England" **File size is approx. 2MB**

### Load, Check and prepare FSA and LSOA data for joins

First, I need to load the cleaned FSA data, followed by the LSOA postcode lookup data, checking both load steps have completed ok.  

In [ ]:
# load and check cleaned FSA data:
import pandas as pd

df = pd.read_csv(
    "../data/processed/fsa_london_establishments_clean.csv",
    dtype={"PostCode": str, "postcode_tier": str} # prevent CSV mis-read on postcode fields
)

print(df.shape)
df.head()

(81217, 28)


,AddressLine1,AddressLine2,AddressLine3,AddressLine4,BusinessName,BusinessType,BusinessTypeID,ChangesByServerID,FHRSID,LocalAuthorityBusinessID,...,RatingKey,RatingValue,RightToReply,SchemeType,geocode.longitude,geocode.latitude,scores.Hygiene,scores.Structural,scores.ConfidenceInManagement,postcode_tier
0,NaN,309 Wood Lane,NaN,Dagenham,5 Elms Cafe,Restaurant/Cafe/Canteen,1,0,1714830,81398,...,fhrs_5_en-gb,5,NaN,FHRS,0.142421,51.554493,5.0,5.0,5.0,full
1,NaN,446 Becontree Avenue,NaN,Dagenham,7 TILL 11,Retailers - other,4613,0,115956,45140,...,fhrs_5_en-gb,5,NaN,FHRS,0.129392,51.558435,5.0,5.0,5.0,full
2,NaN,460 Lodge Avenue,NaN,Dagenham,Aafio Mini Market,Retailers - other,4613,0,1413686,77594,...,fhrs_5_en-gb,5,NaN,FHRS,0.110646,51.534718,5.0,5.0,0.0,full
3,NaN,North Street,NaN,Barking,Abbey Children's Centre Day Nursery,Caring Premises,5,0,128460,58427,...,fhrs_5_en-gb,5,NaN,FHRS,0.075250,51.541375,5.0,5.0,5.0,full
4,NaN,1 Hewett Road,NaN,Dagenham,Abbey Kebab & Pizza,Takeaway/sandwich shop,7844,0,122900,5136,...,fhrs_5_en-gb,5,NaN,FHRS,0.128666,51.550930,5.0,5.0,5.0,full


In [ ]:
# load LSOA lookup file into memory

import pandas as pd

# only two columns are needed from this otherwise v large file
lsoa_lookup = pd.read_csv(
    "../data/raw/postcode_lsoa_lookup.csv",
    usecols=["pcds", "lsoa21cd"],
        # this approach is more efficient than loading the lot and discarding other columns
    dtype=str, # force string (lookup only data but it may look like numeric etc.)
        # and pandas likes to misinterpret code-like columns otherwise
)

# check load ok
print(lsoa_lookup.shape)
print(lsoa_lookup.head())

(2720556, 2)
      pcds   lsoa21cd
0  AB1 0AA  S01013490
1  AB1 0AB  S01013490
2  AB1 0AD  S01013490
3  AB1 0AE  S01013856
4  AB1 0AF  S01013487


-> note the `pcds` column:

Spaces are still preseent in the postcode column. As I will be joining on an exact string-to-string match, spacing between the two join fields needs to be consistent.  

I believe the cleanest approach will be to strip out all spaces, as they are no longer required for classifying postcode format / tiers:

In [ ]:
# create new, formatted postcode column in lookup table:
lsoa_lookup["postcode_key"] = lsoa_lookup["pcds"].str.replace(" ", "", regex=False)

# create equivalent column in FSA table:
df["postcode_key"] = df["PostCode"].str.upper().str.strip().str.replace(" ", "", regex=False)

# check resulting outputs have no spaces:
print(lsoa_lookup["postcode_key"].head())
print(df["postcode_key"].head())

0    AB10AA
1    AB10AB
2    AB10AD
3    AB10AE
4    AB10AF
Name: postcode_key, dtype: str
0     RM83NH
1     RM83UB
2     RM94QS
3    IG118JA
4     RM82XT
Name: postcode_key, dtype: str


-> both outputs are of a matching 'unspaced' format and are ready to join.  

### Join FSA data - LSOA Lookup data

As the FSA data has different 'tiers' of postcode completeness, different approaches are requied for each when joining to the full (outward & inward code) postcodes in the lookup data.

First I will handle the full tier join: